# Exercise 4: computational graphs, again!

### Things we need to agree on

For a feature vector $x$, the model computes $z=\sum_i w_i x_i+b$ and $p=\sigma(z)$. The target $y\in\{0,1\}$ is fixed data. Parameters are leaves with mutable values; constants are leaves that the optimizer never updates.

A node's **children** are its inputs (including parameters). Its **parents** consume its output. A local derivative describes one edge; a global derivative describes the chosen final objective. If one node occurs twice in a parent's child list, both edges contribute.



### Read then Run
Read and run the supplied infrastructure below. You do not need to implement it.

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import List

@dataclass
class ScalarNodeCache:
    value: float = None
    local_partial_derivatives_wrt_children: List[float] = None
    global_derivative_wrt_self: float = None

class ScalarNode:
    def __init__(self, children: List['ScalarNode']) -> None:
        self._parents = []
        self._children = list(children)
        for child in self._children:
            # Parents are unique; repeated CHILD EDGES are retained.
            if self not in child._parents:
                child._parents.append(self)
        self._cache = ScalarNodeCache()

    def value(self):
        raise NotImplementedError()

    def local_partial_derivatives_wrt_children(self):
        raise NotImplementedError()

    def find_self_position_in_parents_children(self, parent):
        # Compatibility helper: first matching edge only.
        for i, child in enumerate(parent._children):
            if child is self:
                return i
        raise ValueError("Node is not a child of this parent")

    def global_derivative_wrt_self(self):
        """Convenience method: use only when the connected graph has one sink."""
        if self._cache.global_derivative_wrt_self is not None:
            return self._cache.global_derivative_wrt_self
        if not self._parents:
            result = 1.0
        else:
            result = 0.0
            for parent in self._parents:
                local = parent.local_partial_derivatives_wrt_children()
                for i, child in enumerate(parent._children):
                    if child is self:
                        result += local[i] * parent.global_derivative_wrt_self()
        self._cache.global_derivative_wrt_self = result
        return result

    def reset_cache(self):
        self._cache = ScalarNodeCache()

    def clean_cache_recursively(self):
        # Graph-aware traversal visits shared nodes once.
        for node in nodes_in_postorder(self):
            node.reset_cache()

    def update_parameters_by_gradient_step(self, learning_rate):
        # Implemented in Task 6.
        return gradient_step(self, learning_rate)

class ConstantNode(ScalarNode):
    def __init__(self, value):
        super().__init__([])
        self._value = float(value)

    def value(self):
        return self._value

class SumNode(ScalarNode):
    def value(self):
        if self._cache.value is None:
            self._cache.value = sum(c.value() for c in self._children)
        return self._cache.value

    def local_partial_derivatives_wrt_children(self):
        return [1.0] * len(self._children)

class ProductNode(ScalarNode):
    def value(self):
        if self._cache.value is None:
            self._cache.value = math.prod(c.value() for c in self._children)
        return self._cache.value

    def local_partial_derivatives_wrt_children(self):
        if self._cache.local_partial_derivatives_wrt_children is None:
            # Multiplying the OTHER values also works when an input is zero.
            self._cache.local_partial_derivatives_wrt_children = [
                math.prod(c.value() for j, c in enumerate(self._children) if j != i)
                for i in range(len(self._children))
            ]
        return self._cache.local_partial_derivatives_wrt_children

def nodes_in_postorder(root):
    """Children before parents, with each node included exactly once (DAGs)."""
    order, seen = [], set()
    def visit(node):
        if node in seen:
            return
        seen.add(node)
        for child in node._children:
            visit(child)
        order.append(node)
    visit(root)
    return order

def invalidate_from(node):
    """Changing a leaf invalidates it and every operation consuming it."""
    stack, seen = [node], set()
    while stack:
        current = stack.pop()
        if current not in seen:
            seen.add(current)
            current.reset_cache()
            stack.extend(current._parents)

def close(actual, expected, message="Unexpected numerical result", atol=1e-9, rtol=1e-7):
    assert math.isclose(actual, expected, abs_tol=atol, rel_tol=rtol), (
        f"{message}: got {actual}, expected {expected}"
    )

def close_list(actual, expected, message="Check list length and child order"):
    assert len(actual) == len(expected), message
    for a, e in zip(actual, expected):
        close(a, e, message)

def must_raise(error_type, callback):
    try:
        callback()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

# Infrastructure smoke checks, including a repeated edge and a zero input.
x = ConstantNode(3)
square = ProductNode([x, x])
close(square.value(), 9)
close(x.global_derivative_wrt_self(), 6)
close_list(ProductNode([ConstantNode(0), ConstantNode(4)]).local_partial_derivatives_wrt_children(), [4, 0])
print("Infrastructure ready.")


## Task 0: Make a parameter mutable

**Why do we need this?** 

Learning changes weights. A constant represents fixed data; a parameter needs a controlled way to change its value without leaving stale computations behind.

A setter must update $w$ and invalidate cached operations that depend on $w$. The helper `invalidate_from` walks towards parents; it is supplied so this warm-up stays small.

**Your task**

Implement `ParameterNode.set_value`: store the new value as a float and call `invalidate_from(self)`. Do not change `ConstantNode`.

<details><summary>Hint: open after trying</summary>

Assign to the same attribute used by `ConstantNode.value()`.

</details>

**Explain after checking:** Why must changing a leaf invalidate a cached sum above it?


In [ ]:
class ParameterNode(ConstantNode):
    def set_value(self, value):
        ### Your Solution starts here ###
        pass
        ### Your Solution ends here ###
        

In [ ]:
p = ParameterNode(2)
c = ConstantNode(3)
s = SumNode([p, c])
close(s.value(), 5)
p.set_value(-1)
close(p.value(), -1)
close(s.value(), 2, "A cached ancestor was not invalidated")
close(c.value(), 3)
assert isinstance(p.value(), float)
p.set_value(0)
close(s.value(), 3)

print("Task 0: all checks passed.")


## Task 1: Compute a linear score

**Why do we need this?** 

A text classifier can give each feature a weight: a positive weight for “excellent”, a negative weight for “awful”, plus a bias for the baseline preference.

The score is $z=\sum_{i=1}^n w_i x_i+b$. It is a real number, not yet a probability. The children are stored in this exact order: `[x_1, ..., x_n, w_1, ..., w_n, b]`.

**Your task**

Implement the uncached calculation in `LinearNode.value`. Cache handling is provided. For zero input features, the output must equal the bias.

<details><summary>Hint: open after trying</summary>

Pair arguments and weights with `zip`, multiply their values, sum, then add the bias value.

</details>

**Explain after checking:** For inputs [2, −1], weights [0.5, 3] and bias 0.2, predict the score before running the check.


In [ ]:
class LinearNode(ScalarNode):
    def __init__(self, arguments, weights, bias):
        if len(arguments) != len(weights):
            raise ValueError("One weight is required per feature")
        super().__init__(list(arguments) + list(weights) + [bias])
        self._arguments = list(arguments)
        self._weights = list(weights)
        self._bias = bias

    def value(self):
        if self._cache.value is not None:
            return self._cache.value
        ### Your Solution starts here ###
        
        ### Your Solution ends here ###
        self._cache.value = result
        return result


In [ ]:
xs = [ConstantNode(2), ConstantNode(-1)]
ws = [ParameterNode(0.5), ParameterNode(3)]
b = ParameterNode(0.2)
z = LinearNode(xs, ws, b)
close(z.value(), -1.8)
close(z.value(), -1.8)
close(LinearNode([], [], ParameterNode(4)).value(), 4)
ws[0].set_value(1.5)
close(z.value(), 0.2)
must_raise(ValueError, lambda: LinearNode(xs, [], b))

print("Task 1: all checks passed.")


## Task 2: Return local derivatives in child order

**Why do we need this?** 

Backpropagation associates each derivative with a child position. Correct numbers in the wrong order silently train the wrong parameters.

For $z=\sum_i w_i x_i+b$: $\partial z/\partial x_i=w_i$, $\partial z/\partial w_i=x_i$, and $\partial z/\partial b=1$. Return one flat list in the constructor’s child order.

**Your task**

Implement `linear_partials` and cache the list in `local_partial_derivatives_wrt_children`. The attachment below installs it as the class method.

<details><summary>Hint — open after trying</summary>

Concatenate the weight values, argument values and `[1.0]`.

</details>

**Explain after checking:** Why is the derivative with respect to the bias independent of its current value?


In [ ]:
def linear_partials(self):
    if self._cache.local_partial_derivatives_wrt_children is not None:
        return self._cache.local_partial_derivatives_wrt_children
    ### Your Solution starts here ###

    ### Your Solution ends here ###
    self._cache.local_partial_derivatives_wrt_children = result
    return result

LinearNode.local_partial_derivatives_wrt_children = linear_partials


In [ ]:
xs = [ConstantNode(2), ConstantNode(-1)]
ws = [ParameterNode(0.5), ParameterNode(3)]
z = LinearNode(xs, ws, ParameterNode(0.2))
close_list(z.local_partial_derivatives_wrt_children(), [0.5, 3, 2, -1, 1])
assert z._cache.local_partial_derivatives_wrt_children is not None
ws[0].set_value(2)
close_list(z.local_partial_derivatives_wrt_children(), [2, 3, 2, -1, 1])
close_list(LinearNode([], [], ParameterNode(9)).local_partial_derivatives_wrt_children(), [1])

print("Task 2: all checks passed.")


## Task 3: Turn scores into probabilities

**Why do we need this?** 

A score such as −1.8 cannot directly be interpreted as a probability. Sigmoid maps every real score to [0, 1] and provides a smooth learning signal.

$\sigma(z)=1/(1+e^{-z})$ and $\sigma\prime(z)=\sigma(z)(1-\sigma(z))$. To avoid overflow, use $1/(1+e^{-z})$ for $z\geq0$ and $e^z/(1+e^z)$ for $z<0$.

**Your task**

Implement the forward value and a one-element list of local derivatives. Reuse `self.value()` for the derivative. Keep both caches consistent.

<details><summary>Hint: open after trying</summary>

Only exponentiate a non-positive number; `math.exp(1000)` overflows.

</details>

**Explain after checking:** Why can a highly confident sigmoid have a very small local derivative?


In [ ]:
class SigmoidNode(ScalarNode):
    def __init__(self, argument):
        super().__init__([argument])

    def value(self):
        if self._cache.value is not None:
            return self._cache.value
        ### Your Solution starts here ###

        ### Your Solution ends here ###
        self._cache.value = result
        return result

    def local_partial_derivatives_wrt_children(self):
        if self._cache.local_partial_derivatives_wrt_children is not None:
            return self._cache.local_partial_derivatives_wrt_children
        ### Your Solution starts here ###

        ### Your Solution ends here ###
        self._cache.local_partial_derivatives_wrt_children = result
        return result


In [ ]:
s = SigmoidNode(ConstantNode(0))
close(s.value(), 0.5)
close_list(s.local_partial_derivatives_wrt_children(), [0.25])
for v in [-1000, -3, 0, 3, 1000]:
    node = SigmoidNode(ConstantNode(v))
    assert 0 <= node.value() <= 1
    assert 0 <= node.local_partial_derivatives_wrt_children()[0] <= 0.25
close(SigmoidNode(ConstantNode(-3)).value() + SigmoidNode(ConstantNode(3)).value(), 1)
x = ParameterNode(0)
s = SigmoidNode(x)
s.value()
s.local_partial_derivatives_wrt_children()
x.set_value(2)
close(s.value(), 1 / (1 + math.exp(-2)))

print("Task 3: all checks passed.")


## Task 4: Audit the chain rule numerically

**Why do we need this?** 

A plausible loss curve cannot prove that gradients are correct. Finite differences give an independent way to catch sign errors and missing factors before training.

Approximate $\frac{\partial f}{\partial\theta}$ with $[f(\theta+h)-f(\theta-h)]/(2h)$. Each perturbation must invalidate caches. Restore the original value even if evaluation raises an exception. Use $h=10^{-5}$ initially; excessively small steps amplify rounding error.

**Your task**

Implement `finite_difference(root, parameter, h)`. Reject non-positive h. Use `try/finally` to restore the parameter. Compare numerical derivatives with `global_derivative_wrt_self()` on a fresh, single-output graph.

<details><summary>Hint: open after trying</summary>

Store the original value; evaluate at plus h and minus h using `set_value`; return their difference divided by `2*h`.

</details>

**Explain after checking:** Why is comparing at a single point weaker evidence than comparing at several nonzero values?


In [ ]:
def finite_difference(root, parameter, h=1e-5):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
for values in [(0.2, -0.7, 0.1), (-1.1, 0.4, -0.3), (0, 0, 0)]:
    w1, w2, b = [ParameterNode(v) for v in values]
    root = SigmoidNode(LinearNode([ConstantNode(2), ConstantNode(-1)], [w1, w2], b))
    for parameter in [w1, w2, b]:
        before = parameter.value()
        numeric = finite_difference(root, parameter)
        root.clean_cache_recursively()
        analytic = parameter.global_derivative_wrt_self()
        close(analytic, numeric, "Chain-rule gradient mismatch", atol=1e-7, rtol=1e-5)
        close(parameter.value(), before, "Gradient checking changed a parameter")
must_raise(ValueError, lambda: finite_difference(root, b, 0))
class ExplodingNode(ScalarNode):
    def value(self):
        raise RuntimeError("Deliberate failure")
p = ParameterNode(7)
must_raise(RuntimeError, lambda: finite_difference(ExplodingNode([p]), p))
close(p.value(), 7, "Parameter was not restored after an exception")

print("Task 4: all checks passed.")


## Task 5: Build binary cross-entropy

**Why do we need this?** 

We need an objective that rewards the probability assigned to the correct label and penalizes confident mistakes more strongly than uncertain ones.

For a fixed binary target, $\ell(p,y)=-y\log p-(1-y)\log(1-p)$ and $\partial\ell/\partial p=-y/p+(1-y)/(1-p)$. Use `math.log1p(-p)` for the second log. This probability-based loss requires $0<p<1$: reject endpoints rather than clipping them and silently changing the derivative.

**Your task**

Implement the value and local derivative. Validate y in {0,1} at construction and validate p at evaluation. The target is fixed metadata; only the probability is a differentiable child.

<details><summary>Hint: open after trying</summary>

The derivative list has one entry. For y=1, increasing p should reduce the loss.

</details>

**Explain after checking:** Derive the composed derivative with respect to z and show that it simplifies to p−y.


In [ ]:
class CrossEntropyLoss(ScalarNode):
    def __init__(self, y_hat, gold_label):
        if gold_label.value() not in (0, 1):
            raise ValueError("The gold label must be binary")
        super().__init__([y_hat])
        self._gold_label = gold_label

    def _probability(self):
        p = self._children[0].value()
        if not 0 < p < 1:
            raise ValueError("Probability-based BCE requires 0 < p < 1; use logits BCE for extremes")
        return p

    def value(self):
        if self._cache.value is not None:
            return self._cache.value
        ### Your Solution starts here ###

        ### Your Solution ends here ###
        self._cache.value = result
        return result

    def local_partial_derivatives_wrt_children(self):
        if self._cache.local_partial_derivatives_wrt_children is not None:
            return self._cache.local_partial_derivatives_wrt_children
        ### Your Solution starts here ###

        ### Your Solution ends here ###
        self._cache.local_partial_derivatives_wrt_children = result
        return result


In [ ]:
for y in (0, 1):
    for p_value in (0.1, 0.5, 0.9):
        p = ParameterNode(p_value)
        loss = CrossEntropyLoss(p, ConstantNode(y))
        expected = -math.log(p_value if y else 1-p_value)
        close(loss.value(), expected)
        analytic = loss.local_partial_derivatives_wrt_children()[0]
        close(analytic, finite_difference(loss, p), atol=1e-6)
    z = ParameterNode(0.8)
    p = SigmoidNode(z)
    loss = CrossEntropyLoss(p, ConstantNode(y))
    close(z.global_derivative_wrt_self(), p.value() - y)
must_raise(ValueError, lambda: CrossEntropyLoss(ConstantNode(0.5), ConstantNode(2)))
for endpoint in [0, 1, -0.1, 1.1]:
    loss = CrossEntropyLoss(ConstantNode(endpoint), ConstantNode(1))
    must_raise(ValueError, loss.value)
    must_raise(ValueError, loss.local_partial_derivatives_wrt_children)

print("Task 5: all checks passed.")


## Task 6: Take one simultaneous gradient step

**Why do we need this?** 

Gradient descent updates every parameter using the gradient at the same old parameter vector. Updating during recursive traversal can change later gradients or update a shared parameter twice.

The rule is $\theta_i^{new}=\theta_i^{old}-\eta\,\frac{\partial L}{\partial\theta_i}$. Compute all gradients first, then apply updates. Reset all objective caches before differentiation and after updating. For this task the graph must have exactly one terminal output.

**Your task**

Implement `gradient_step(root, learning_rate)`: reject negative rates, collect unique ParameterNodes with `nodes_in_postorder`, snapshot all new values, then call their setters. Return nothing. Use the provided parent-based derivative method for now.

<details><summary>Hint: open after trying</summary>

A list of `(parameter, new_value)` pairs prevents partial updates from affecting later gradient calculations.

</details>

**Explain after checking:** For L=a·b at a=2, b=3 and η=0.1, why must b become 2.8 rather than 2.83?


In [ ]:
def gradient_step(root, learning_rate):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
a, b = ParameterNode(2), ParameterNode(3)
root = ProductNode([a, b])
gradient_step(root, 0.1)
close(a.value(), 1.7)
close(b.value(), 2.8, "Updates were not simultaneous")
x = ParameterNode(2)
root = ProductNode([x, x])
root.value()
x.global_derivative_wrt_self()
root.update_parameters_by_gradient_step(0.1)
close(x.value(), 1.6, "A shared parameter was updated more than once")
close(root.value(), 2.56, "Stale forward cache after update")
close(x.global_derivative_wrt_self(), 3.2, "Stale gradient cache after update")
w, b = ParameterNode(0), ParameterNode(0)
fixed = ConstantNode(2)
loss = CrossEntropyLoss(SigmoidNode(LinearNode([fixed], [w], b)), ConstantNode(1))
before = loss.value()
gradient_step(loss, 0.1)
assert loss.value() < before, "A small descent step should reduce this loss"
close(fixed.value(), 2)
must_raise(ValueError, lambda: gradient_step(loss, -0.1))

print("Task 6: all checks passed.")


## Task 7: Backpropagate from an explicit output

**Why do we need this?** 

In a batch, the same weight participates in several losses. A parameter may also be connected to an old evaluation graph. “Follow all parents” no longer specifies which objective to differentiate.

Reverse-mode differentiation starts with $\frac{\partial L}{\partial L}=1$. Visit the selected root’s reachable nodes in reverse postorder and accumulate $g_{child} \mathrel{+}=g_{node}\,\frac{\partial \text{node}}{\partial \text{child}}$. The dictionary is fresh for every call, and repeated child positions are separate edges.

**Your task**

Implement `backward(root)` returning a dictionary from each reachable node to its derivative. Clear root caches first. Follow child links only and accumulate rather than overwrite. Then implement the marked portion of `gradient_step` using this dictionary. This replaces the single-output optimizer from Task 6 for all remaining tasks.

<details><summary>Hint: open after trying</summary>

Initialize every reachable gradient to zero, seed root to one, then loop over `reversed(order)` and `enumerate(node._children)`.

</details>

**Explain after checking:** Why should a disconnected objective sharing a weight contribute exactly zero to the selected root’s derivative?


In [ ]:
def backward(root):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def gradient_step(root, learning_rate):
    if learning_rate < 0:
        raise ValueError("Learning rate must be non-negative")
    ### Your Solution starts here ###

    ### Your Solution ends here ###


In [ ]:
x = ParameterNode(2)
chosen = ProductNode([x, x])
unrelated = ProductNode([ConstantNode(100), x])
g = backward(chosen)
close(g[chosen], 1)
close(g[x], 4)
assert unrelated not in g
close(backward(unrelated)[x], 100)
close(backward(chosen)[x], 4, "Gradients leaked across backward calls")
# Diamond: both branches use a shared intermediate.
x = ParameterNode(3)
u = ProductNode([x, ConstantNode(2)])
root = SumNode([ProductNode([u, u]), u])  # 4*x*x + 2*x
close(backward(root)[x], 26)
close(backward(root)[x], finite_difference(root, x), atol=1e-7)
# An unrelated parent must not affect the optimizer either.
x = ParameterNode(2)
root = ProductNode([x, x])
noise = ProductNode([x, ConstantNode(100)])
gradient_step(root, 0.1)
close(x.value(), 1.6)
close(noise.value(), 160)
# Two distinct parameters still update simultaneously.
a, b = ParameterNode(2), ParameterNode(3)
gradient_step(ProductNode([a, b]), 0.1)
close_list([a.value(), b.value()], [1.7, 2.8])

print("Task 7: all checks passed.")


## Task 8: Fuse sigmoid and loss for numerical stability

**Why do we need this?** 

A confident wrong prediction should produce a large, finite loss and a strong gradient. Explicit sigmoid followed by log can instead round to log(0).

For finite logits z, use $\ell(z,y)=\max(z,0)-yz+\log(1+e^{-|z|})$ and $\Large{\frac{\partial\ell}{\partial z}}=\sigma(z)-y$. Use `math.log1p`. This is the same mathematical objective as Task 5; its floating-point behavior is better. Do not clip probabilities. The target remains fixed metadata.

**Your task**

Implement `BinaryCrossEntropyWithLogits.value` and its one-element derivative. Compute sigmoid with the stable branch from Task 3, without adding temporary graph nodes. Reject non-binary labels.

<details><summary>Hint: open after trying</summary>

The exponential in the loss always has a non-positive argument. For z=1000 and y=0, the loss should be approximately 1000 and the derivative approximately 1.

</details>

**Explain after checking:** Why can the loss gradient stay large even when a standalone sigmoid derivative is nearly zero?


In [ ]:
class BinaryCrossEntropyWithLogits(ScalarNode):
    def __init__(self, logit, gold_label):
        if gold_label.value() not in (0, 1):
            raise ValueError("The gold label must be binary")
        super().__init__([logit])
        self._gold_label = gold_label

    def value(self):
        if self._cache.value is not None:
            return self._cache.value
        ### Your Solution starts here ###

        ### Your Solution ends here ###
        self._cache.value = result
        return result

    def local_partial_derivatives_wrt_children(self):
        if self._cache.local_partial_derivatives_wrt_children is not None:
            return self._cache.local_partial_derivatives_wrt_children
        ### Your Solution starts here ###

        ### Your Solution ends here ###
        self._cache.local_partial_derivatives_wrt_children = result
        return result


In [ ]:
for y in (0, 1):
    for v in (-5, -0.8, 0, 0.8, 5):
        z = ParameterNode(v)
        stable = BinaryCrossEntropyWithLogits(z, ConstantNode(y))
        plain = CrossEntropyLoss(SigmoidNode(z), ConstantNode(y))
        close(stable.value(), plain.value())
        close(backward(stable)[z], backward(plain)[z])
        close(backward(stable)[z], finite_difference(stable, z), atol=1e-7)
    for v in (-1000, 1000):
        z = ParameterNode(v)
        loss = BinaryCrossEntropyWithLogits(z, ConstantNode(y))
        assert math.isfinite(loss.value())
        expected_loss = 1000 if (v > 0) != bool(y) else 0
        close(loss.value(), expected_loss)
        close(backward(loss)[z], (1.0 if v > 0 else 0.0) - y)
must_raise(ValueError, lambda: BinaryCrossEntropyWithLogits(ConstantNode(0), ConstantNode(0.5)))

print("Task 8: all checks passed.")


## Task 9: Construct a mean batch objective

**Why do we need this?** 

A model learns one shared set of weights from many examples. Averaging the losses keeps the update scale comparable when the batch size changes.

For N examples, $L=\frac{1}{N}\sum_j\ell(z_j,y_j)$. Every example must reuse the identical `ParameterNode` objects. Feature values and labels become fresh constants. Return a single scalar objective; its graph should contain each weight object once in the unique-node traversal.

**Your task**

Implement `make_batch_loss(features, labels, weights, bias)`. Reject empty batches, mismatched example/label counts and incorrect feature dimensions. Build the mean using the supplied SumNode and ProductNode plus Task 8’s loss.

<details><summary>Hint: open after trying</summary>

A mean is a sum multiplied by a `ConstantNode(1/N)`. Do not copy parameters or convert them into constants.

</details>

**Explain after checking:** What happens to the gradient if every example is duplicated and you use a sum instead of a mean?


In [ ]:
def make_batch_loss(features, labels, weights, bias):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
X, y = [[1, 0], [0, 2], [-1, 1]], [1, 0, 1]
w = [ParameterNode(0.2), ParameterNode(-0.3)]
b = ParameterNode(0.1)
loss = make_batch_loss(X, y, w, b)
doubled = make_batch_loss(X + X, y + y, w, b)
close(loss.value(), doubled.value())
g1, g2 = backward(loss), backward(doubled)
for p in w + [b]:
    close(g1[p], g2[p], "Mean gradient changed when examples were duplicated")
    close(g1[p], finite_difference(loss, p), atol=1e-7)
reachable_parameters = [n for n in nodes_in_postorder(loss) if isinstance(n, ParameterNode)]
assert len(reachable_parameters) == 3
assert set(reachable_parameters) == set(w + [b])
must_raise(ValueError, lambda: make_batch_loss([], [], w, b))
must_raise(ValueError, lambda: make_batch_loss(X, [1], w, b))
must_raise(ValueError, lambda: make_batch_loss([[1]], [1], w, b))
must_raise(ValueError, lambda: make_batch_loss([[1, 2]], [2], w, b))

print("Task 9: all checks passed.")


### A tiny NLP setting

Our features are `[count("excellent"), count("awful")]`; positive sentiment is label 1. This deliberately simple bag-of-words representation ignores word order, negation and every other token. It makes the learning mechanics visible, but cannot represent all meanings.

The fixed examples below are synthetic teaching data. Perfect accuracy on them demonstrates that the implementation can fit a simple pattern; it is not evidence of real-world sentiment performance. We build one reusable training graph so repeated epochs do not attach a new set of parent links every time.


In [ ]:
def text_features(text):
    tokens = text.lower().split()
    return [tokens.count("excellent"), tokens.count("awful")]

TRAIN_TEXTS = ["excellent", "excellent excellent", "excellent story", "awful", "awful awful", "awful story"]
TRAIN_X = [text_features(t) for t in TRAIN_TEXTS]
TRAIN_Y = [1, 1, 1, 0, 0, 0]
PROBE_TEXTS = ["excellent excellent story", "awful awful story", "excellent awful", "not excellent"]
PROBE_X = [text_features(t) for t in PROBE_TEXTS]
print(list(zip(TRAIN_TEXTS, TRAIN_X, TRAIN_Y)))


## Task 10 : Train and inspect a text classifier

**Why do we need this?** 

An optimizer is useful only if the whole loop works repeatedly: evaluate the objective, differentiate, update and evaluate again. Predictions then connect abstract parameters back to text.

Use full-batch gradient descent with learning rate 0.2 for 200 steps, starting at zero. `history[0]` is the initial loss and each following entry is the loss after one update. For prediction, compute z numerically and a stable sigmoid; classify p≥0.5 as positive. Prediction must not attach new nodes to training parameters.

**Your task**

Implement `train` and `predict_probabilities`. Return `(weights, bias, history)` from train and a list of probabilities from predict. Require nonnegative steps and learning rate. Build the training objective once outside the epoch loop. Report initial/final loss, learned parameters and probe probabilities.

<details><summary>Hint: open after trying</summary>

Call `gradient_step` once per iteration. Each step handles caches. For prediction, use parameter `.value()` results in ordinary arithmetic.

</details>

**Explain after checking:** Explain the score for “excellent awful”. Why does the model classify “not excellent” incorrectly?


In [ ]:
def train(features, labels, steps=200, learning_rate=0.2):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###

def predict_probabilities(features, weights, bias):
    ### Your Solution starts here ###
    pass
    ### Your Solution ends here ###


In [ ]:
trained_w, trained_b, history = train(TRAIN_X, TRAIN_Y)
assert len(history) == 201
assert all(math.isfinite(v) for v in history)
close(history[0], math.log(2))
assert history[-1] < 0.12, "Loss did not fall enough on the toy data"
assert all(after <= before + 1e-12 for before, after in zip(history, history[1:])), (
    "With this dataset and learning rate, each step should reduce the loss"
)
train_probs = predict_probabilities(TRAIN_X, trained_w, trained_b)
assert [int(p >= 0.5) for p in train_probs] == TRAIN_Y
assert trained_w[0].value() > 0 and trained_w[1].value() < 0
parent_counts = [len(p._parents) for p in trained_w + [trained_b]]
probe_probs = predict_probabilities(PROBE_X, trained_w, trained_b)
assert parent_counts == [len(p._parents) for p in trained_w + [trained_b]]
assert probe_probs[0] > 0.9 and probe_probs[1] < 0.1
close(probe_probs[2], 0.5)
assert probe_probs[3] > 0.5  # Known representation limitation, not a success metric.
_, _, zero_history = train(TRAIN_X, TRAIN_Y, steps=0)
assert len(zero_history) == 1
must_raise(ValueError, lambda: train(TRAIN_X, TRAIN_Y, steps=-1))
must_raise(ValueError, lambda: train(TRAIN_X, TRAIN_Y, learning_rate=-1))
must_raise(ValueError, lambda: predict_probabilities([[1]], trained_w, trained_b))
print(f"Loss: {history[0]:.4f} → {history[-1]:.4f}")
print("Weights:", [round(w.value(), 4) for w in trained_w], "bias:", round(trained_b.value(), 4))
for text, probability in zip(PROBE_TEXTS, probe_probs):
    print(f"{text!r}: p(positive)={probability:.3f}")

print("Task 10: all checks passed.")
